In [1]:
import numpy as np
import pandas as pd
import uproot
from IPython.display import display


In [2]:
MASS = 0.02045833  # GeV
EPSILON = 0.01
ALPHA = 1 / 137

In [3]:
meson = pd.read_csv("../2x2MCP-PythiaGen/outputs/meson_summary_50pt.csv")
dy = pd.read_csv("../2x2MCP-DY/outputs/production/aggregate_summary.csv")

In [4]:
parents = {
    111: (r"$\pi^0$", 0.1349768, 0.98823, 0),
    221: (r"$\eta$", 0.5478620, 0.39410, 0),
    331: (r"$\eta'$", 0.9577800, 0.0220, 0),
    113: (r"$\rho$", 0.7752600, 4.72e-5, 1),
    223: (r"$\omega$", 0.7826500, 7.36e-5, 1),
    333: (r"$\phi$", 1.0194610, 2.973e-4, 1),
    443: (r"$J/\psi$", 3.0969000, 5.971e-2, 1),
}

In [5]:
sigma = meson[meson["sigma_gen_mb_mean"] > 0]
SIGMA_SOFT_MB = sigma.loc[
    sigma["production_mode"] == 0, "sigma_gen_mb_mean"
].mean()
charm_scale = sigma.loc[
    sigma["production_mode"] == 1, "sigma_gen_mb_mean"
].mean() / SIGMA_SOFT_MB

table = {}
for pdg, (label, parent_mass, br_ref, kind) in parents.items():
    d = meson[(meson["emitter_pdg"] == pdg)
              & np.isclose(meson["mcp_mass_GeV"], MASS)]

    r = (MASS / parent_mass)**2
    phase = (max(0, 1 - 4*r)**3 if kind == 0
             else np.sqrt(max(0, 1 - 4*r)) * (1 + 2*r))
    br = EPSILON**2 * ALPHA * br_ref * phase
    scale = charm_scale if pdg == 443 else 1
    yield_per_pot = scale * d["n_emitter_total"].sum() / d["n_events_generated"].sum()
    table[label] = [yield_per_pot, br, 2 * yield_per_pot * br]

In [6]:
d = dy[np.isclose(dy["mcp_mass_GeV"], MASS)].iloc[0]
yield_per_pot = EPSILON**2 * d["sigma_over_epsilon2_mb"] / SIGMA_SOFT_MB
table["DY"] = [yield_per_pot, np.nan, 2 * yield_per_pot]

In [7]:
table = pd.DataFrame(table, index=["#/POT", "BR", "MCP/POT"])
print(f"mχ = {MASS:g} GeV, ε = {EPSILON:g}")
display(table.style.format("{:.4e}", na_rep=""))

mχ = 0.0204583 GeV, ε = 0.01


,$\pi^0$,$\eta$,$\eta'$,$\rho$,$\omega$,$\phi$,$J/\psi$,DY
#/POT,2.8568e+00,3.2001e-01,3.3912e-02,3.6596e-01,3.6766e-01,1.0979e-02,5.4487e-07,1.3517e-12
BR,5.4019e-07,2.8288e-07,1.5971e-08,3.4452e-11,5.3722e-11,2.1701e-10,4.3584e-08,
MCP/POT,3.0864e-06,1.8105e-07,1.0832e-09,2.5216e-11,3.9504e-11,4.7651e-12,4.7495e-14,2.7034e-12


In [8]:
table.to_csv("production_table.csv")

geometry = meson.loc[
    np.isclose(meson["mcp_mass_GeV"], MASS),
    ["emitter_pdg", "acceptance_fraction"]
].copy()
geometry = pd.concat([
    geometry,
    dy.loc[np.isclose(dy["mcp_mass_GeV"], MASS),
           ["emitter_pdg", "acceptance_fraction"]]
], ignore_index=True)
geometry.to_csv("geometry_acceptance.csv", index=False)